<a href="https://colab.research.google.com/github/rizkyhaksono/llm-vs-slm-lab/blob/main/02-inference-perbandingan/01_hello_groq_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02.01 — Hello Groq LLM

**Tujuan**: panggilan pertama ke LLM via API (Groq). Belajar pattern dasar: completion, streaming, hitung token, hitung biaya, eksperimen temperature.

**Prasyarat**: `00-setup-dan-tools/01_setup_environment.ipynb` lulus (API key ke-set, baik via `.env` lokal atau Colab Secrets).

## 0. Bootstrap (jalankan pertama)

In [1]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break
assert repo_root is not None, "Tidak ketemu root repo (requirements.txt)."
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"IN_COLAB={IN_COLAB}, repo_root={repo_root}")

Cloning into 'llm-vs-slm-lab'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 83 (delta 21), reused 62 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 143.92 KiB | 2.40 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/llm-vs-slm-lab
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 MB 11.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 

## 1. Panggilan pertama (non-streaming)

Pattern paling dasar: kirim prompt → tunggu seluruh response → cetak.

In [2]:
import time

from utils.llm_clients import GROQ_DEFAULT_MODEL, groq_client

client = groq_client()

t0 = time.perf_counter()
resp = client.chat.completions.create(
    model=GROQ_DEFAULT_MODEL,  # "llama-3.1-8b-instant"
    messages=[
        {"role": "system", "content": "Kamu asisten yang menjawab singkat dan jelas dalam Bahasa Indonesia."},
        {"role": "user", "content": "Apa ibu kota Indonesia? Jelaskan dalam 1 kalimat."},
    ],
    temperature=0,
    max_tokens=80,
)
elapsed_ms = (time.perf_counter() - t0) * 1000

print(f"Response: {resp.choices[0].message.content}")
print(f"\nLatency:        {elapsed_ms:.0f} ms")
print(f"Input tokens:   {resp.usage.prompt_tokens}")
print(f"Output tokens:  {resp.usage.completion_tokens}")
print(f"Model:          {resp.model}")

Response: Ibu kota Indonesia adalah Jakarta, yang merupakan kota terbesar dan pusat pemerintahan negara.

Latency:        1632 ms
Input tokens:   70
Output tokens:  26
Model:          llama-3.1-8b-instant


## 2. Streaming — token sampai sebelum jawaban penuh

Streaming bikin UX terasa lebih cepat: user mulai membaca sebelum LLM selesai generate seluruh response.

Metrik penting baru: **Time To First Token (TTFT)** — berapa lama dari kirim request sampai token pertama datang.

In [3]:
t0 = time.perf_counter()
ttft_ms = None
chunks: list[str] = []

stream = client.chat.completions.create(
    model=GROQ_DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Tuliskan 3 fakta menarik tentang Borobudur dalam Bahasa Indonesia."}],
    temperature=0,
    max_tokens=200,
    stream=True,
)

print("Response (streaming):\n")
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    if delta and ttft_ms is None:
        ttft_ms = (time.perf_counter() - t0) * 1000
    print(delta, end="", flush=True)
    chunks.append(delta)

total_ms = (time.perf_counter() - t0) * 1000
print(f"\n\nTTFT:          {ttft_ms:.0f} ms")
print(f"Total latency: {total_ms:.0f} ms")
print(f"Chars total:   {sum(len(c) for c in chunks)}")

Response (streaming):

Berikut 3 fakta menarik tentang Borobudur:

1. **Borobudur adalah salah satu situs warisan dunia UNESCO**: Borobudur merupakan salah satu situs warisan dunia UNESCO yang terletak di Magelang, Jawa Tengah. Situs ini diakui sebagai salah satu keajaiban dunia karena keindahan dan keunikan arsitekturnya.

2. **Borobudur dibangun pada abad ke-9**: Borobudur dibangun pada abad ke-9 oleh Dinasti Syailendra, sebuah kerajaan Hindu yang berkuasa di Jawa Tengah pada saat itu. Situs ini awalnya bernama Vajrasana, yang berarti "throne dari baja" dalam bahasa Sanskerta.

3. **Borobudur memiliki 2.672 relief dan 504 patung Buddha**: Borobudur memiliki lebih dari 2

TTFT:          108 ms
Total latency: 414 ms
Chars total:   657


**Insight**: TTFT biasanya jauh lebih kecil dari total latency. Inilah kenapa streaming kelihatan "snappier" walaupun total waktu hampir sama.

## 3. Hitung biaya per panggilan

Harga Groq llama-3.1-8b-instant (per Mei 2026): ~$0.05/1M input token, ~$0.08/1M output token. Untuk panggilan kecil seperti di atas, biayanya **sangat kecil** — tapi di scale 100k request/hari, ini jadi nyata.

Cek harga terkini: https://groq.com/pricing

In [4]:
from utils.benchmark import estimate_cost_usd

# pakai usage dari panggilan section 1
in_tok = resp.usage.prompt_tokens
out_tok = resp.usage.completion_tokens

cost_one = estimate_cost_usd(in_tok, out_tok)
print(f"Biaya 1 panggilan: ${cost_one:.8f}")
print(f"\nKalau di-scale:")
for n in [100, 1_000, 10_000, 100_000, 1_000_000]:
    print(f"  {n:>9,} request → ${cost_one * n:>10.2f}")

Biaya 1 panggilan: $0.00558000

Kalau di-scale:
        100 request → $      0.56
      1,000 request → $      5.58
     10,000 request → $     55.80
    100,000 request → $    558.00
  1,000,000 request → $   5580.00


## 4. Temperature: deterministik vs kreatif

Parameter `temperature` kontrol seberapa "acak" sampling token berikutnya:
- `0` = greedy (selalu pilih token paling probable) → output **deterministik & monoton**.
- `0.7-1.0` = balanced → output **variatif & natural**.
- `> 1` = makin acak → makin kreatif tapi sering ngawur.

Demo: prompt sama, jalankan 3x di temp 0 (output identik) dan 3x di temp 0.8 (output beda-beda).

In [5]:
def quick_call(prompt: str, temperature: float) -> str:
    r = client.chat.completions.create(
        model=GROQ_DEFAULT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=40,
    )
    return r.choices[0].message.content.strip()

prompt = "Berikan 1 ide nama startup AI di Indonesia, satu nama saja, satu baris."

print("=== temperature=0 (deterministik) ===")
for i in range(3):
    print(f"  Run {i+1}: {quick_call(prompt, 0)}")

print("\n=== temperature=0.8 (kreatif) ===")
for i in range(3):
    print(f"  Run {i+1}: {quick_call(prompt, 0.8)}")

=== temperature=0 (deterministik) ===
  Run 1: Nama startup AI di Indonesia: "Pintar Indonesia"
  Run 2: Nama startup AI di Indonesia: "Pintar Indonesia"
  Run 3: Nama startup AI di Indonesia: "Pintar Indonesia"

=== temperature=0.8 (kreatif) ===
  Run 1: Namasoft AI Indonesia.
  Run 2: Nama startup AI di Indonesia: "Ayuza"
  Run 3: "Mindaura"


## Refleksi & insight

1. **API call sederhana**: kirim messages → dapat response. Boilerplate-nya nyaris nol.
2. **Streaming** tidak bikin total lebih cepat, tapi bikin user **persepsi**-nya cepat. Untuk chatbot, wajib pakai.
3. **Token bukan kata**: 1 kata Bahasa Indonesia rata-rata 1.5–2 token (boros dibanding English).
4. **Biaya per request kecil**, tapi scale × konstanta. Untuk task yang volumenya tinggi (mis. sentiment analysis 100k komentar), self-hosted SLM bisa lebih murah.
5. **Temperature 0** dipakai saat butuh reproducibility (benchmark, klasifikasi). Temperature ~0.7 dipakai untuk konten kreatif / variatif.

## Latihan mandiri

1. Coba ganti model di `GROQ_DEFAULT_MODEL` jadi `llama-3.3-70b-versatile` (model lebih besar). Bandingkan latency dan kualitas jawaban. Berapa lipat lebih lambat? Apakah kualitas signifikan lebih baik untuk prompt simple?
2. Buat function `chat_loop()` yang loop minta input user, kirim ke Groq, cetak response. Pakai streaming. Pertahankan history percakapan.

## Lanjut

Sekarang kita pindah ke sisi SLM — load model lokal di CPU: [02_hello_local_slm.ipynb](02_hello_local_slm.ipynb)